<a href="https://colab.research.google.com/github/fatemarampurawalaa-sudo/LLM-practice/blob/main/FastAPI_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q fastapi uvicorn

print("Setup done ✅")

Setup done ✅


In [ ]:
%%writefile main.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Hello! Property API is running"}

@app.get("/properties/{location}")
def get_location(location: str):
    return {"location": location, "status": "We have properties here"}

Writing main.py


In [ ]:
import subprocess, time

server = subprocess.Popen(["uvicorn", "main:app", "--port", "8000"])
time.sleep(3)
print("Server started ✅")

Server started ✅


In [ ]:
import requests

r1 = requests.get("http://127.0.0.1:8000/")
print(r1.json())

r2 = requests.get("http://127.0.0.1:8000/properties/Marina")
print(r2.json())

{'message': 'Hello! Property API is running'}
{'location': 'Marina', 'status': 'We have properties here'}


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, path="/docs", height=600)

<IPython.core.display.Javascript object>

In [ ]:
%%writefile main.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from google import genai
from google.genai import types

app = FastAPI(title="Property AI API")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
CHAT_MODEL = "gemini-flash-lite-latest"

class Question(BaseModel):
    question: str

class Answer(BaseModel):
    answer: str

@app.get("/")
def home():
    return {"message": "Hello! Property API is running"}

@app.post("/ask", response_model=Answer)
def ask(q: Question):
    response = client.models.generate_content(
        model=CHAT_MODEL,
        contents=q.question,
        config=types.GenerateContentConfig(
            system_instruction="You are a Dubai real estate assistant. Only answer property questions. Answer in maximum 3 sentences.",
            max_output_tokens=300
        )
    )
    return Answer(answer=response.text)

Overwriting main.py


In [ ]:
import os, subprocess, time, requests
from google.colab import userdata

server.terminate()
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

server = subprocess.Popen(["uvicorn", "main:app", "--port", "8000"])
time.sleep(5)

r = requests.post("http://127.0.0.1:8000/ask", json={"question": "What is a good area in Dubai for families?"})
print(r.status_code)
print(r.json())

200
{'answer': 'Dubai Hills Estate is an exceptional area for families, offering extensive parks, top-tier schools, and a family-friendly community atmosphere centered around an 18-hole championship golf course. Another fantastic option is Arabian Ranches, known for its spacious townhouses and villas, community pools, and secure, gated environment. Both locations provide a perfect blend of tranquility, modern amenities, and easy access to central Dubai.'}


In [ ]:
bad = requests.post("http://127.0.0.1:8000/ask", json={"text": "Hello"})
print(bad.status_code)
print(bad.json())

422
{'detail': [{'type': 'missing', 'loc': ['body', 'question'], 'msg': 'Field required', 'input': {'text': 'Hello'}}]}
